In [1]:
# Session 1 — LLM Mechanics and Prompt Engineering

In [2]:
# How an LLM behaves and how we can control it through good prompting before we start building RAG or AI Agents.

# Problem statement:
# Imagine Walmart wants to use an LLM in real production applications—for customer support, product classification, supply-chain decisions, incident handling, supplier communication, etc.
# The problem is:
#     How do we communicate with an LLM so that it gives consistent, useful, cost-efficient and machine-readable answers instead of unpredictable free-form responses?

In [3]:
# LLM mechanics:
# 1. Tokenization
# 2. Context Window
# 3. Temperature

# Prompt Engineering:
# 1. Zero-shot
# 2. Few-shot
# 3. Chain-of-Thought
# 4. Role Prompting
# 5. Structured Output

# Big picture of this notebook:
#     Understand how the LLM works → learn how to control it with prompts → force reliable structured outputs → apply everything to a realistic Walmart production use case.

# And this notebook intentionally prepares the foundation for the next notebook on RAG pipelines.

## Setup — Load Libraries and API Keys


In [4]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import tiktoken

load_dotenv(override=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("OK Libraries loaded. OpenAI client ready.")
print(f"   Using model: gpt-4o-mini (cost-efficient for labs)")
print(f"   Using model: gpt-4o (for advanced demonstrations)")


OK Libraries loaded. OpenAI client ready.
   Using model: gpt-4o-mini (cost-efficient for labs)
   Using model: gpt-4o (for advanced demonstrations)


---
# Part 1: LLM Mechanics

Before we build production systems with LLMs, we need to understand exactly how they work — not at a research level, but at the level that matters for engineering decisions.

Three mechanics that every AI engineer must understand:
1. **Tokenization** — What the model actually reads
2. **Context Window** — What the model can remember
3. **Temperature** — How the model chooses its next word


## 1.1 Tokenization

An LLM does not read words. It reads **tokens**.

A token is roughly 3–4 characters in English. Common words are one token. Rare words, proper nouns, and code identifiers are often split into multiple tokens.

**Why this matters for engineering:**
- API costs are charged per token, not per word
- Context window limits are measured in tokens
- Poorly structured prompts waste tokens and increase cost

Let's see exactly how Walmart-relevant text gets tokenized.


In [5]:
# LLMs do not read text as complete words. They first break the text into tokens, and token count directly affects context usage and API cost.

# Tokenization Demo
enc = tiktoken.encoding_for_model("gpt-4o")

texts = [
    "Walmart",
    "Walmart processes 2.3 million customer transactions per hour.",
    "Supply chain optimization using LangGraph and LLM-as-judge evaluation.",
    "SKU-7829341-B",   # A product identifier — watch how this is split
    "def run_agent(state: AgentState) -> AgentState:",  # Code
]

print(f"Tokenizer: {enc.name}")
print(f"Vocabulary size: {enc.n_vocab:,} tokens")
print()
print(f"{'Text':<55} | {'Tokens':>6} | {'Token IDs (first 8)'}")
print("-" * 90)

for text in texts:
    token_ids = enc.encode(text)
    token_words = [enc.decode([t]) for t in token_ids]
    preview_ids = str(token_ids[:8]) + ("..." if len(token_ids) > 8 else "")
    print(f"{text[:54]:<55} | {len(token_ids):>6} | {preview_ids}")

print()
print("Key insight: 'SKU-7829341-B' is split into multiple tokens.")
print("Structured identifiers cost more tokens than natural text.")


Tokenizer: o200k_base
Vocabulary size: 200,019 tokens

Text                                                    | Tokens | Token IDs (first 8)
------------------------------------------------------------------------------------------
Walmart                                                 |      2 | [54, 40173]
Walmart processes 2.3 million customer transactions pe  |     13 | [54, 40173, 14340, 220, 17, 13, 18, 5749]...
Supply chain optimization using LangGraph and LLM-as-j  |     14 | [49473, 13464, 34658, 2360, 27830, 9922, 326, 451]...
SKU-7829341-B                                           |      6 | [72089, 12, 47574, 47717, 16, 8287]
def run_agent(state: AgentState) -> AgentState:         |     12 | [1314, 2461, 62035, 16114, 25, 28237, 1881, 8]...

Key insight: 'SKU-7829341-B' is split into multiple tokens.
Structured identifiers cost more tokens than natural text.


### Token Cost Calculator

At $0.150 per 1M input tokens (gpt-4o-mini), even small inefficiencies in prompts add up at Walmart's scale.


In [6]:
# Token Cost Calculator
def estimate_cost(text: str, model: str = "gpt-4o-mini", direction: str = "input") -> dict:
    """Estimate the token count and API cost for a given text."""
    enc = tiktoken.encoding_for_model("gpt-4o")  # gpt-4o enc works for gpt-4o-mini too
    token_count = len(enc.encode(text))
    
    # Pricing as of 2026 (approximate, verify at platform.openai.com)
    pricing = {
        "gpt-4o-mini": {"input": 0.150, "output": 0.600},   # per 1M tokens
        "gpt-4o":      {"input": 2.50,  "output": 10.00},
    }
    
    price_per_million = pricing.get(model, {}).get(direction, 0)
    cost_usd = (token_count / 1_000_000) * price_per_million
    
    return {
        "text_chars": len(text),
        "token_count": token_count,
        "chars_per_token": round(len(text) / token_count, 2) if token_count > 0 else 0,
        "cost_usd": round(cost_usd, 8),
        "cost_per_1000_calls": round(cost_usd * 1000, 4),
    }

# Compare prompt efficiency
prompts = {
    "Verbose system prompt": """You are an AI assistant working at Walmart. Your job is to help customer service 
    representatives answer questions. You should always be helpful, professional, and accurate. You must never 
    make up information that you do not know. Always be polite and courteous to customers.""",
    
    "Concise system prompt": """You are a Walmart customer service AI. Be accurate, professional, brief.""",
}

print("Prompt Efficiency Comparison (gpt-4o-mini, input tokens):")
print("-" * 70)
for name, prompt in prompts.items():
    stats = estimate_cost(prompt, "gpt-4o-mini", "input")
    print(f"\n{name}:")
    print(f"  Characters : {stats['text_chars']}")
    print(f"  Tokens     : {stats['token_count']}")
    print(f"  Cost/call  : ${stats['cost_usd']:.6f}")
    print(f"  Cost/1000 calls: ${stats['cost_per_1000_calls']}")

print()
print("At 1M calls/day (Walmart scale), the difference compounds significantly.")


Prompt Efficiency Comparison (gpt-4o-mini, input tokens):
----------------------------------------------------------------------

Verbose system prompt:
  Characters : 284
  Tokens     : 55
  Cost/call  : $0.000008
  Cost/1000 calls: $0.0083

Concise system prompt:
  Characters : 72
  Tokens     : 15
  Cost/call  : $0.000002
  Cost/1000 calls: $0.0023

At 1M calls/day (Walmart scale), the difference compounds significantly.


## 1.2 Context Window

The context window is the model's working memory — everything it can "see" at once.

| Model | Context Window |
|---|---|
| gpt-4o-mini | 128,000 tokens |
| gpt-4o | 128,000 tokens |
| Claude Sonnet | 200,000 tokens |

**Why this matters:**
- Long documents must be chunked if they exceed the window
- Every message in a chat adds to the context and costs tokens
- Agents with long tool call histories can hit context limits mid-task

Let's demonstrate how context window enables multi-turn memory.


In [7]:
# Context Window Demo — Multi-turn Memory
print("=== Demo: How Context Window Creates Memory ===\n")

# Conversation 1: Without context history (model has no memory)
print("NO Without context (no message history passed):")
response_no_context = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What inventory threshold did I mention?"}
    ],
    max_tokens=80,
    temperature=0
)
print(f"   Model says: {response_no_context.choices[0].message.content.strip()}\n")

# Conversation 2: With full context history passed
print("OK With context (full message history passed):")
conversation_history = [
    {"role": "user",      "content": "Our reorder threshold for SKU-7829 is 500 units."},
    {"role": "assistant", "content": "Understood. The reorder threshold for SKU-7829 is 500 units."},
    {"role": "user",      "content": "And for SKU-9102 it's 250 units."},
    {"role": "assistant", "content": "Got it. SKU-9102 reorder threshold is 250 units."},
    {"role": "user",      "content": "What inventory threshold did I mention for SKU-7829?"},
]

response_with_context = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=conversation_history,
    max_tokens=80,
    temperature=0
)
print(f"   Model says: {response_with_context.choices[0].message.content.strip()}\n")

# Show context size
enc = tiktoken.encoding_for_model("gpt-4o")
total_tokens = sum(len(enc.encode(m["content"])) for m in conversation_history)
print(f"STATS Context used: ~{total_tokens} tokens")
print(f"   Window capacity: 128,000 tokens")
print(f"   Remaining: {128000 - total_tokens:,} tokens")


=== Demo: How Context Window Creates Memory ===

NO Without context (no message history passed):
   Model says: I'm sorry, but I don't have access to previous conversations or specific details about your inventory threshold. If you provide me with more context or details, I would be happy to help you with your inquiry!

OK With context (full message history passed):
   Model says: You mentioned that the inventory reorder threshold for SKU-7829 is 500 units.

STATS Context used: ~66 tokens
   Window capacity: 128,000 tokens
   Remaining: 127,934 tokens


## 1.3 Temperature

Temperature controls **randomness** in the model's output.

| Temperature | Behaviour | Use Case |
|---|---|---|
| 0.0 | Deterministic, always most likely token | Classification, extraction, code generation |
| 0.2–0.5 | Slightly varied, mostly consistent | Summaries, factual Q&A, structured outputs |
| 0.7–1.0 | Creative, more varied | Marketing copy, ideation, creative writing |
| > 1.2 | Highly random, may be incoherent | Rarely useful in production |

**Rule of thumb for production systems:** Use the lowest temperature that produces acceptable output quality.


In [8]:
# Temperature Demo
import time

prompt = "In one sentence, suggest how Walmart can use AI to improve store checkout speed."

print("=== Temperature Effect on Output Variability ===\n")
print(f"Prompt: \"{prompt}\"\n")

for temp in [0.0, 0.5, 1.2]:
    print(f"Temperature = {temp}:")
    responses = []
    for i in range(3):
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=60,
        )
        responses.append(r.choices[0].message.content.strip())
        time.sleep(0.3)  # Avoid rate limiting
    
    for i, resp in enumerate(responses, 1):
        print(f"  Run {i}: {resp}")
    print()

print("Observation: At temp=0.0, all 3 runs give identical output.")
print("At temp=1.2, outputs diverge significantly — unpredictable for production.")


=== Temperature Effect on Output Variability ===

Prompt: "In one sentence, suggest how Walmart can use AI to improve store checkout speed."

Temperature = 0.0:
  Run 1: Walmart can implement AI-powered checkout systems that utilize computer vision and machine learning to automatically scan and process items in customers' carts, significantly reducing wait times at the register.
  Run 2: Walmart can implement AI-powered checkout systems that utilize computer vision and machine learning to automatically scan and process items in customers' carts, significantly reducing wait times at checkout.
  Run 3: Walmart can implement AI-powered checkout systems that utilize computer vision and machine learning to automatically scan and process items in customers' carts, significantly reducing wait times at checkout.

Temperature = 0.5:
  Run 1: Walmart can implement AI-powered automated checkout systems that utilize computer vision and machine learning to streamline the scanning process and reduce

In [9]:
# Temperature Demo
import time

prompt = "Write a short poem on my dog Jeff who loves to play fetch in the park."

print("=== Temperature Effect on Output Variability ===\n")
print(f"Prompt: \"{prompt}\"\n")

for temp in [0.0, 0.5, 1.2]:
    print(f"Temperature = {temp}:")
    responses = []
    for i in range(3):
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=60,
        )
        responses.append(r.choices[0].message.content.strip())
        time.sleep(0.3)  # Avoid rate limiting
    
    for i, resp in enumerate(responses, 1):
        print(f"  Run {i}: {resp}")
    print()

print("Observation: At temp=0.0, all 3 runs give identical output.")
print("At temp=1.2, outputs diverge significantly — unpredictable for production.")


=== Temperature Effect on Output Variability ===

Prompt: "Write a short poem on my dog Jeff who loves to play fetch in the park."

Temperature = 0.0:
  Run 1: In the golden light of day,  
Jeff bounds through the park to play,  
With a wagging tail and joyful bark,  
He’s the king of fetch, our little spark.  

A ball in sight, he takes his stance,  
With a gleam in his eye, he starts
  Run 2: In the golden light of day,  
Jeff bounds through the park to play,  
With a wagging tail and joyful bark,  
He’s the king of fetch, our little spark.  

A ball in sight, he takes his stance,  
With a gleam in his eye, he starts
  Run 3: In the golden light of day,  
Jeff bounds through the park to play,  
With a wagging tail and joyful bark,  
He’s the king of fetch, our little spark.  

A ball in sight, he takes his stance,  
With a gleam in his eye, he starts

Temperature = 0.5:
  Run 1: In the golden glow of the park’s embrace,  
Runs my dear Jeff, with joy on his face.  
A ball in his mouth

---
# Part 2: Prompt Engineering

Prompt engineering is the practice of designing inputs to LLMs to consistently produce the outputs you need.

It is not a workaround. It is a core engineering discipline. A well-engineered prompt can often outperform a fine-tuned model — at a fraction of the cost.

We will cover 5 patterns, each with a Walmart business use case.


In [10]:
# Imagine you walk into a restaurant and tell the waiter:
#     “Bring me food.”
# “What do you think the waiter will bring you?”

# But if you say:
#     “Please get me a medium-rare grilled paneer steak, no onions, extra lemon on the side, served hot with garlic naan and Coke Zero.”
# The waiter knows exactly what to do.


## 2.1 Zero-Shot Prompting

Ask the model to perform a task with no examples. Works well when:
- The task is common in the model's training data
- The output format is simple

**Walmart use case:** Classify the urgency of a customer support ticket.


In [11]:
# Zero-Shot Prompting
def zero_shot_classify(complaint: str) -> str: #  "My item arrived damaged and I need a replacement immediately."
    """Classify complaint severity with no examples — pure zero-shot."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a Walmart customer support triage system. "
                    "Classify each complaint as CRITICAL, HIGH, MEDIUM, or LOW severity. "
                    "Reply with exactly one word: the severity level."
                )
            },
            {"role": "user", "content": f"Complaint: {complaint}"}
        ],
        temperature=0,
        max_tokens=10,
    )
    return response.choices[0].message.content.strip()

complaints = [
    "My item arrived damaged and I need a replacement immediately.",
    "Website was slow this morning but seems fine now.",
    "I was charged twice for the same order — my bank is disputing it.",
    "The packaging could be more eco-friendly.",
    "I can't log into my Walmart+ account and my groceries won't deliver.",
]

print("Zero-Shot Complaint Classification:")
print("-" * 65)
for c in complaints:
    severity = zero_shot_classify(c)
    emoji = {"CRITICAL": "CRITICAL", "HIGH": "HIGH", "MEDIUM": "MEDIUM", "LOW": "LOW"}.get(severity, "UNKNOWN")
    print(f"{emoji} {severity:<10} | {c[:55]}")


Zero-Shot Complaint Classification:
-----------------------------------------------------------------
CRITICAL CRITICAL   | My item arrived damaged and I need a replacement immedi
LOW LOW        | Website was slow this morning but seems fine now.
CRITICAL CRITICAL   | I was charged twice for the same order — my bank is dis
LOW LOW        | The packaging could be more eco-friendly.
CRITICAL CRITICAL   | I can't log into my Walmart+ account and my groceries w


## 2.2 Few-Shot Prompting

Provide examples of input → output pairs before the actual task. The model learns the pattern from your examples.

**When to use over zero-shot:**
- You need a specific output format
- The task has domain-specific nuances the model might not know
- Zero-shot gives inconsistent results

**Walmart use case:** Extract structured data from unstructured customer feedback.


In [ ]:
# Few-Shot Prompting
# Chain-of-Thought Prompting
# Role Prompting
# Structured Output

# RAG Architecture